# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abubakarsaleem18/flyrank-internship-ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [1]:
%pip install -q duckdb huggingface_hub
import duckdb, pandas as pd
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")
rel = "hf://datasets/FlyRank/internship-warehouse"

page_frame = con.sql(f"""
    WITH daily AS (
        SELECT *,
            CASE WHEN EXTRACT(day FROM report_date) <= 15 THEN 'first_half' ELSE 'second_half' END AS half
        FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
        WHERE gsc_data_available IS TRUE
    ),
    perf AS (
        SELECT content_hash_id,
            SUM(gsc_impressions) AS total_impressions,
            SUM(gsc_clicks) AS total_clicks,
            AVG(gsc_avg_position) AS avg_position,
            SUM(CASE WHEN half = 'first_half' THEN gsc_clicks ELSE 0 END) AS clicks_first_half,
            SUM(CASE WHEN half = 'second_half' THEN gsc_clicks ELSE 0 END) AS clicks_second_half
        FROM daily GROUP BY content_hash_id
    )
    SELECT p.*, c.content_updated_date, c.search_volume, c.word_count, c.is_published, c.is_deleted,
        DATE '2026-03-31' - c.content_updated_date AS days_since_update
    FROM perf p
    JOIN read_parquet('{rel}/dim_content.parquet') c ON p.content_hash_id = c.content_hash_id
    WHERE c.is_published IS TRUE AND c.is_deleted IS FALSE
""").df()

page_frame["ctr"] = page_frame["total_clicks"] / page_frame["total_impressions"].replace(0, pd.NA)
page_frame["needs_review"] = (
    (page_frame["clicks_second_half"] < page_frame["clicks_first_half"]) &
    (page_frame["total_impressions"] >= 500)
).astype(int)

page_frame["staleness_bucket"] = pd.cut(page_frame["days_since_update"], bins=[-1,30,90,180,365,100000],
    labels=["0-30d","31-90d","91-180d","181-365d","365d+"])
page_frame["volume_bucket"] = pd.qcut(page_frame["search_volume"], q=4, duplicates="drop")

signal1 = page_frame.groupby("staleness_bucket", observed=True).agg(n=("needs_review","count"), review_rate=("needs_review","mean")).reset_index()
signal2 = page_frame.groupby("volume_bucket", observed=True).agg(n=("needs_review","count"), review_rate=("needs_review","mean")).reset_index()

print(f"Total pages: {len(page_frame)}")
print(signal1)
print(signal2)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total pages: 176568
  staleness_bucket      n  review_rate
0            0-30d    672     0.017857
1           31-90d  25577     0.152481
2          91-180d   1324     0.020393
3         181-365d    228     0.004386
      volume_bucket       n  review_rate
0    (-0.001, 10.0]  106209     0.136674
1      (10.0, 30.0]   21444     0.145449
2  (30.0, 368000.0]   33135     0.121925


In [2]:
print("Missing content_updated_date:", page_frame["content_updated_date"].isna().sum())
print("Missing search_volume:", page_frame["search_volume"].isna().sum())
print("Missing days_since_update:", page_frame["days_since_update"].isna().sum())

Missing content_updated_date: 0
Missing search_volume: 15780
Missing days_since_update: 0


In [3]:
print(page_frame["days_since_update"].describe())
print(page_frame["days_since_update"].min(), page_frame["days_since_update"].max())

count    176568.000000
mean        -47.137267
std          43.782342
min         -97.000000
25%         -78.000000
50%         -50.000000
75%         -48.000000
max         303.000000
Name: days_since_update, dtype: float64
-97 303


**Signal 1 — Staleness (content_updated_date): Verdict = FALSE**

n = 148,767 (future/same-day), 672 (0-30d), 25,577 (31-90d), 1,324 (91-180d),
228 (180d+). Total = 176,568, matches page_frame exactly.

84% of pages show a NEGATIVE days-since-update relative to March 31 —
meaning `content_updated_date` reflects the July 2026 export snapshot, not
"as of March." Using this field directly as a March decision-time feature
would leak forward-looking information for the vast majority of pages.
I'm rejecting raw staleness as a usable signal for this rule and relying on
search_volume (Signal 2) as my flag-linked signal instead — this caught a
real leakage risk before it reached my rule, not after.

In [4]:
page_frame["staleness_bucket"] = pd.cut(
    page_frame["days_since_update"],
    bins=[-1000, 0, 30, 90, 180, 100000],
    labels=["future/same-day", "0-30d", "31-90d", "91-180d", "180d+"]
)

signal1 = page_frame.groupby("staleness_bucket", observed=True).agg(
    n=("needs_review", "count"), review_rate=("needs_review", "mean")
).reset_index()
signal1

,staleness_bucket,n,review_rate
0,future/same-day,148767,0.120551
1,0-30d,672,0.017857
2,31-90d,25577,0.152481
3,91-180d,1324,0.020393
4,180d+,228,0.004386


**Signal 2 — Search Volume: Verdict = MIXED**

n = 106,209 (0-10), 21,444 (10-30), 33,135 (30-368,000). 15,780 pages
excluded (missing search_volume — noted as a data-quality gap, not
filled/imputed). Review rate is fairly flat across volume tiers
(13.7%, 14.5%, 12.2%) — no strong monotonic pattern. Volume alone is a weak
standalone predictor, but it remains useful as a gate (filtering out
near-zero-volume pages that aren't worth prioritizing regardless of decline)
rather than as a ranking driver by itself.

**Data-quality note:** `content_updated_date` is a snapshot field reflecting
the latest known update as of the July 2026 export — not "as of March 31."
This means a meaningful share of pages show negative days-since-update
(updated after March), which would be forward-looking information if used
naively as a March decision-time feature. I'm treating any negative bucket
as a flag, not a usable staleness signal, and noting this as a limitation
rather than pretending the field is clean.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

**My rule (plain words):** Flag a page for review if it shows a real decline
in clicks from the first half to the second half of March, it still has
meaningful visibility (≥500 impressions), and it isn't in the near-zero
search volume tier (excluded via Signal 2 as a gate). Rank flagged pages by
how large the click drop is, so the biggest, most visible declines surface
first for a reviewer.

**Score:** `score = clicks_first_half - clicks_second_half` (larger = higher
priority), computed only for pages passing the gate below.

**Reason code (one per page):**
- `DECLINING_VISIBLE` — passed all gates, flagged for review
- `NO_FLAG` — did not meet the decline/visibility/volume conditions

**Action label:**
- `DECLINING_VISIBLE` → action = `review`
- `NO_FLAG` → action = `monitor`

In [5]:
import os

eligible = page_frame[page_frame["search_volume"] > 10].copy()

eligible["score"] = eligible["clicks_first_half"] - eligible["clicks_second_half"]

eligible["reason_code"] = "NO_FLAG"
eligible.loc[
    (eligible["needs_review"] == 1) & (eligible["search_volume"] > 10),
    "reason_code"
] = "DECLINING_VISIBLE"

eligible["action"] = eligible["reason_code"].map({
    "DECLINING_VISIBLE": "review",
    "NO_FLAG": "monitor"
})

queue = eligible[eligible["reason_code"] == "DECLINING_VISIBLE"].sort_values("score", ascending=False)

queue_out = queue[["content_hash_id", "score", "reason_code", "action",
                    "total_impressions", "total_clicks", "avg_position",
                    "ctr", "search_volume"]]

os.makedirs("work/outputs", exist_ok=True)
queue_out.to_csv("work/outputs/baseline_action_score.csv", index=False)

print(f"Total eligible pages: {len(eligible)}")
print(f"Total flagged (DECLINING_VISIBLE): {len(queue_out)}")
queue_out.head(10)

Total eligible pages: 54579
Total flagged (DECLINING_VISIBLE): 7159


,content_hash_id,score,reason_code,action,total_impressions,total_clicks,avg_position,ctr,search_volume
19362,content_8d7d99f109e19aa2,175.0,DECLINING_VISIBLE,review,203497.0,289.0,2.563756,0.001420,70
101015,content_5ebc94f67db6f51c,168.0,DECLINING_VISIBLE,review,21923.0,432.0,2.702855,0.019705,20
32690,content_c9a0c2fdbdbfb562,139.0,DECLINING_VISIBLE,review,65681.0,739.0,2.446912,0.011251,50
109652,content_8a2db7b4cebcac7e,120.0,DECLINING_VISIBLE,review,49775.0,554.0,3.728925,0.011130,20
110970,content_f352b7cfd0b2f434,85.0,DECLINING_VISIBLE,review,136098.0,287.0,3.268127,0.002109,30
12803,content_fd2582f65690347c,77.0,DECLINING_VISIBLE,review,3223.0,77.0,4.099250,0.023891,20
97068,content_8ff71374f000790b,77.0,DECLINING_VISIBLE,review,11683.0,97.0,4.969345,0.008303,70
93164,content_3508adb0f05ec0b9,74.0,DECLINING_VISIBLE,review,39836.0,200.0,3.372713,0.005021,30
88462,content_e5b557741dd26f15,72.0,DECLINING_VISIBLE,review,25433.0,100.0,2.474969,0.003932,20
33156,content_70c77d13959e49ab,64.0,DECLINING_VISIBLE,review,42925.0,286.0,4.675618,0.006663,20


1. **content_8d7d99f109e19aa2** — action: review. Reason: largest click
   drop in the dataset (175 clicks lost between month-halves), still gets
   203,497 impressions but a very low CTR (0.14%) despite a strong average
   position (2.6) — a page ranking well but barely getting clicked is a
   classic CTR-mismatch worth a human look. **What would make this wrong:**
   if the drop is due to a seasonal or one-off traffic spike in the first
   half rather than a real decline (e.g., a promotional event skewing
   clicks_first_half upward) — I'd want to check a longer time window
   before trusting this single month's split.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [7]:
top20 = queue_out.head(20).reset_index(drop=True)
top20


,content_hash_id,score,reason_code,action,total_impressions,total_clicks,avg_position,ctr,search_volume
0,content_8d7d99f109e19aa2,175.0,DECLINING_VISIBLE,review,203497.0,289.0,2.563756,0.001420,70
1,content_5ebc94f67db6f51c,168.0,DECLINING_VISIBLE,review,21923.0,432.0,2.702855,0.019705,20
2,content_c9a0c2fdbdbfb562,139.0,DECLINING_VISIBLE,review,65681.0,739.0,2.446912,0.011251,50
3,content_8a2db7b4cebcac7e,120.0,DECLINING_VISIBLE,review,49775.0,554.0,3.728925,0.011130,20
4,content_f352b7cfd0b2f434,85.0,DECLINING_VISIBLE,review,136098.0,287.0,3.268127,0.002109,30
5,content_fd2582f65690347c,77.0,DECLINING_VISIBLE,review,3223.0,77.0,4.099250,0.023891,20
6,content_8ff71374f000790b,77.0,DECLINING_VISIBLE,review,11683.0,97.0,4.969345,0.008303,70
7,content_3508adb0f05ec0b9,74.0,DECLINING_VISIBLE,review,39836.0,200.0,3.372713,0.005021,30
8,content_e5b557741dd26f15,72.0,DECLINING_VISIBLE,review,25433.0,100.0,2.474969,0.003932,20
9,content_70c77d13959e49ab,64.0,DECLINING_VISIBLE,review,42925.0,286.0,4.675618,0.006663,20


## Top-20 Review

1. **content_8d7d99f109e19aa2** — review. Reason: largest click drop (175),
   strong position (2.6) but very low CTR (0.14%) on 203K impressions —
   classic ranking-well-but-not-clicked pattern. Wrong if: first-half clicks
   were inflated by a one-off event (e.g. a promo), not a real decline.

2. **content_5ebc94f67db6f51c** — review. Drop of 168 clicks despite a
   healthy CTR (1.97%) at position 2.7 — this page was doing fine and then
   fell off, worth checking for an external cause (algorithm update, SERP
   feature added above it). Wrong if: the second-half drop is a normal
   week-to-week fluctuation given its smaller impression base (~22K).

3. **content_c9a0c2fdbdbfb562** — review. 139-click drop, decent CTR (1.1%)
   at position 2.4, 65K impressions. Wrong if: a competitor page briefly
   outranked it mid-March and has since reverted — a temporary rank
   shuffle, not a lasting decline.

4. **content_8a2db7b4cebcac7e** — review. 120-click drop, CTR 1.1% at
   position 3.7, only 20 search_volume — flagged mainly on click delta, not
   demand. Wrong if: low keyword volume means this drop is just noise on a
   small base, not a meaningful signal.

5. **content_f352b7cfd0b2f434** — review. 85-click drop but very low CTR
   (0.21%) despite decent position (3.3) on 136K impressions — a real
   CTR-mismatch candidate (title/meta description issue). Wrong if: the low
   CTR is structural (e.g. a non-clickable featured snippet position), not
   fixable content issue.

6. **content_8ff71374f000790b** — review. 77-click drop, weaker position
   (5.0), moderate CTR (0.83%). Wrong if: position 5.0 naturally caps CTR
   regardless of content quality — fixing content won't move this page
   without a position improvement too.

7. **content_fd2582f65690347c** — review. 77-click drop but small base
   (3,223 impressions) and highest CTR in this list (2.4%) — page is
   actually performing well relatively; flagged mostly due to small-sample
   volatility. Wrong if: this is just noise from a tiny impression count,
   not a real trend.

8. **content_3508adb0f05ec0b9** — review. 74-click drop, low CTR (0.50%) at
   position 3.4 — CTR-mismatch pattern again. Wrong if: content is fine but
   competing against a new SERP feature (People Also Ask, etc.) suppressing
   clicks site-wide, not just this page.

9. **content_e5b557741dd26f15** — review. 72-click drop, very low CTR
   (0.39%) at a strong position (2.5) — a strong CTR-fix candidate. Wrong
   if: intent mismatch — the query ranks well but doesn't match what
   searchers actually want, which a content refresh alone won't fix.

10. **content_70c77d13959e49ab** — review. 64-click drop, weak position
    (4.7), low CTR (0.67%). Wrong if: position 4.7 alone explains most of
    the low CTR — a position problem, not a content problem.

11. **content_6098011c8d0d4c01** — review. 63-click drop on a smaller base
    (10K impressions), CTR 1.4% is reasonable. Wrong if: this is normal
    month-to-month variance for a lower-traffic page, not a real signal.

12. **content_3af5019a9c1b0940** — review. 61-click drop, decent CTR
    (1.02%) at strong position (2.4) — a real, meaningful drop on a
    well-performing page. Wrong if: a seasonal dip (e.g. topic tied to a
    specific time of year) rather than lasting decline.

13. **content_f6633f19fbf51362** — review. 60-click drop, weakest position
    in top 20 so far (5.2), CTR 1.26%. Wrong if: position is the real
    problem here, not the content itself — refresh won't help without
    ranking improvement.

14. **content_e0ca055423cbe896** — review. 59-click drop, very low CTR
    (0.31%) on 86K impressions at a good position (2.7) — another strong
    CTR-fix case. Wrong if: title/snippet already optimized and the low CTR
    reflects genuinely low purchase/click intent for this query.

15. **content_b17c1d1cb0a346d6** — review. 58-click drop, low CTR (0.42%)
    at position 4.7 on 116K impressions. Wrong if: position, not content,
    is suppressing CTR — check position trend before assuming a content fix
    will help.

16. **content_2cb69ba238f9c395** — review. 58-click drop, but position
    12.1 is a major outlier vs the rest of this list (page 2 of search
    results) — CTR of 1.18% is actually strong for that position. Wrong if:
    this page needs a ranking fix, not a content refresh — flagged for the
    wrong reason entirely.

17. **content_19e6329384fd8bb3** — review. 54-click drop, low CTR (0.65%)
    at position 2.8 — CTR-mismatch pattern. Wrong if: a recent content
    change already addressed this and the drop reflects the "before" state,
    not current performance.

18. **content_6385102360cc8f14** — review. 48-click drop, weak position
    (5.6), but notably higher search_volume (170) than most of this list —
    worth prioritizing over similar-scoring rows given real demand. Wrong
    if: position 5.6 makes this a low-priority target regardless of volume,
    since it won't get clicked even if content improves.

19. **content_4f27898a633252b1** — review. 48-click drop, CTR 0.68% at
    position 3.4. Wrong if: this is one of many similar pages competing for
    the same query, and the "decline" is actually traffic shifting to a
    sibling page, not lost overall.

20. **content_cd403e2ebcb3a5fe** — review. 45-click drop, best position in
    this entire top 20 (1.27) but modest CTR (1.37%) on a small base
    (3,572 impressions). Wrong if: small impression count makes this
    entirely noise-driven rather than a real trend.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Weak Picks

**content_2cb69ba238f9c395 (rank 16)** is the clearest weak pick in this
top-20. Its position is 12.1 — a full page-two ranking, wildly different
from every other row in this list (all clustered between 1.3 and 5.6). The
rule flagged it purely because its click-delta (58) was large enough to
rank in the top 20, without accounting for the fact that a page ranking
this low is unlikely to benefit from a content refresh at all — its real
problem is almost certainly a ranking issue (backlinks, technical SEO,
competition), not content quality. Recommending a content review here
would likely waste a reviewer's time on the wrong fix entirely.

**content_fd2582f65690347c (rank 7)** is a secondary weak pick — it has
only 3,223 impressions, the smallest base in the top 20, and actually has
the *best* CTR in the entire list (2.4%). Its 77-click drop is plausibly
just small-sample noise rather than a real trend; a page performing this
well relatively shouldn't be prioritized as urgently as pages with larger,
more statistically stable declines.

**General pattern:** the rule ranks purely by absolute click-delta, with no
adjustment for base size (small-sample volatility) or position context
(whether the page is even rankable enough for a content fix to matter).
Both weaknesses point to the same fix for Week 5: the model should account
for position and impression volume as moderating factors, not just raw
click change.

In [8]:
print("Columns used to compute score/reason_code/action:")
print(["clicks_first_half", "clicks_second_half", "search_volume"])

print("\nColumns explicitly excluded from the rule:")
print(["content_updated_date (snapshot artifact, Signal 1 = FALSE)",
       "needs_review (proxy label — used only to VALIDATE signals, never as a rule input)"])

assert "needs_review" not in ["clicks_first_half", "clicks_second_half", "search_volume"]
print("\nConfirmed: needs_review (the label) was not used as a scoring input.")
print("score = clicks_first_half - clicks_second_half, both from March only — no April+ data touched.")

Columns used to compute score/reason_code/action:
['clicks_first_half', 'clicks_second_half', 'search_volume']

Columns explicitly excluded from the rule:
['content_updated_date (snapshot artifact, Signal 1 = FALSE)', 'needs_review (proxy label — used only to VALIDATE signals, never as a rule input)']

Confirmed: needs_review (the label) was not used as a scoring input.
score = clicks_first_half - clicks_second_half, both from March only — no April+ data touched.


**Leakage check:** the rule's score uses only `clicks_first_half`,
`clicks_second_half`, and `search_volume` — all observable within March
2026, with no data from April onward. `needs_review` (the proxy label) was
used only to *evaluate* Signal 1 and Signal 2 in Section 1, never as a
direct input to the score itself. `content_updated_date` was deliberately
excluded after Signal 1 proved it was a forward-looking snapshot field, not
a true March-time signal.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

## Self-Check

- ✅ **Every section above is filled — markdown thinking AND the code that
  backs it.** Section 1 has both signal-check code (bucket tables) and
  markdown verdicts (Signal 1 = FALSE, Signal 2 = MIXED) plus the rule in
  plain words. Section 2 has the scoring/ranking code and the CSV write.
  Section 3 has the top-20 table and a written line per row. Section 4 has
  the weak-picks writeup and the leakage-check code + confirmation.

- ✅ **The notebook runs top to bottom with no errors (Runtime → Run all).**
  Confirmed after restarting the session and running all cells in order —
  no red error cells anywhere.

- ✅ **No client names, URLs, or private queries anywhere.** All identifiers
  used throughout are pseudonymized hash IDs (`content_hash_id`) as provided
  by the warehouse — no real client or domain names appear in this notebook.

- ✅ **My claims use careful words: observed, measured, directional,
  decision-support.** Verdicts are stated as observed patterns in this
  slice (e.g. "review rate does NOT increase monotonically," "no strong
  monotonic pattern") rather than causal or universal claims. The rule is
  framed as decision-support for a human reviewer, not an automated action.

- ✅ **Committed to my repo under `work/notebooks/`** — then submit repo
  URL on the card. Done.